# Day 5 Project — Mini AI Harness
Everything from 5.1 to 5.6 in one place. One runtime hosts two agent configurations, a
registry supplies scoped tools, policy governs them, events explain the runs, a checkpoint
carries an approval, and MCP sits behind the same boundary as everything else.

Then you add a third agent — and change no runtime code at all.


## Before you begin

### Learning outcomes

- Run two different agents through a single runtime, registry and policy.
- Follow one approval from pause to checkpoint to resolution.
- Add a third agent configuration without editing a single line of the harness.

Architecture reference: [Day 5 diagrams D16-D18](../diagrams/source/day_05.md).

### Expected observation

The research agent completes with tool evidence; the task agent pauses on an external action; a newly written third configuration produces its own tool requests and policy events.


## Concept briefing

## Mapping the course to production systems

| Course term | Common production terminology |
|---|---|
| Provider adapter | model client/provider layer |
| Agent configuration | agent definition/profile |
| Harness runtime | agent runtime/orchestration layer |
| Tool registry | tool/plugin registry |
| Policy | authorization or guardrail middleware |
| Events | tracing/telemetry |
| Checkpoint store | durable execution/state persistence |
| MCP client | protocol integration layer |

Production SDKs package different subsets of these responsibilities. Students should be
able to open an unfamiliar SDK and locate where its model calls, tools, policy, state and
events live rather than assuming the SDK itself is the architecture.

## What the mini harness does not provide

The classroom harness is intentionally not a production platform. It does not provide
enterprise identity, operating-system sandboxing, remote MCP authentication, distributed
workers, deployment or guaranteed model quality. Its purpose is to make the essential
boundaries visible so students can recognise and evaluate larger systems later.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/mini_harness"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

# 3) Day 5 helper: agent configurations are DATA. Read one from configs/<name>.json
#    and turn it into the AgentConfig dataclass the runtime expects.
import json
from mini_harness import AgentConfig, ModelConfig

def load_config(name):
    raw = json.loads((PROJECT_ROOT / "configs" / f"{name}.json").read_text(encoding="utf-8"))
    raw["model"] = ModelConfig(**raw["model"])   # nested dict -> nested dataclass
    return AgentConfig(**raw)

print("Configs      :", sorted(p.stem for p in (PROJECT_ROOT / "configs").glob("*.json")))

## Step 1 — One runtime, two agents

Build the harness once. Everything after this reuses these three objects.


In [ ]:
from mini_harness import EventStore, HarnessRuntime, MockModel, build_demo_registry

registry = build_demo_registry()
events = EventStore()
runtime = HarnessRuntime(registry, MockModel(), events)

print("Registry holds :", [s.name for s in registry.discover()])
print("Runtime object :", type(runtime).__name__)
print()

for name, prompt in [("research_agent", "What is a harness?"),
                     ("task_agent", "Prepare a concise project update")]:
    config = load_config(name)
    result = runtime.run(config, prompt)
    print(f"--- {name} ---")
    print("  visible tools:", [s.name for s in registry.discover(config.allowed_tools)])
    print("  status       :", result.status)
    print("  output       :", result.output)
    print("  events       :", [e["event"] for e in result.events])
    print()

## Step 2 — The approval path, end to end

The external action pauses, the checkpoint holds the exact arguments, and a separate call
carries the human decision.


In [ ]:
task = load_config("task_agent")
pending = runtime.run(task, "Send a synthetic project update")

print("Status        :", pending.status)
print("Approval card :")
print("  tool     :", pending.pending_action["tool"])
print("  arguments:", pending.pending_action["arguments"])
print("  requested by agent:", pending.pending_action["agent"], "at step",
      pending.pending_action["steps_used"])

final = runtime.resume(pending.run_id, task, approved=True)
print()
print("After approval:", final.status)
print("Tool output   :", final.output)
print("Trace         :", [e["event"] for e in final.events])

## Step 3 — The two small pieces we have not used yet

A memory interface and an MCP boundary. Both are deliberately minimal: their job is to
show *where* the responsibility sits, not to be good at it.


In [ ]:
from mini_harness import FakeMCPClient, SimpleMemory, tool_result_payload

memory = SimpleMemory()
memory.add("fictional_asha", "Prefer concise project updates")
memory.add("fictional_asha", "Lab reports are due on Fridays")
print("Memory recall for 'concise update':", memory.search("fictional_asha", "concise update"))

client = FakeMCPClient()
discovered = await client.list_tools()
print("MCP discovery:", [t.name for t in discovered])
print("MCP call     :", tool_result_payload(await client.call_tool("course_lookup",
                                                                  {"topic": "harness"})))
print()
print("Both sit OUTSIDE the loop. Neither can bypass policy to cause a side effect.")

## Step 4 — What the whole day looks like as one trace

Every event recorded by the shared `EventStore`, grouped by run.


In [ ]:
print(f"Runs recorded in this EventStore: {len(events.by_run)}")
print()
for run_id, rows in events.by_run.items():
    agent = rows[0]["details"].get("agent", "?")
    print(f"run {run_id[:8]}  agent={agent}")
    for row in rows:
        detail = row["details"]
        note = detail.get("decision") or detail.get("tool") or detail.get("error") or ""
        print(f"    {row['event']:<20} {note}")
    print()

### Try it yourself

Add a **third** agent configuration — one that looks something up and then wants to email
the result — without editing `runtime.py`, `policy.py` or `providers.py`.


In [ ]:
# --- Worked solution ---
# A configuration is DATA, so a third agent is a new JSON document. Nothing in
# src/mini_harness/ changes. The `mock_plan` field is what lets the deterministic
# mock model act on a config it has never seen before.
import json

notes_agent_json = json.dumps({
    "name": "notes_agent",
    "instructions": "Look up a course note, then request that it be emailed.",
    "allowed_tools": ["lookup_notes", "send_email"],   # a read tool and an external one
    "max_steps": 4,
    "model": {"provider": "mock", "model": "mock-deterministic",
              "temperature": 0.0, "max_output_tokens": 300},
    "mock_plan": [
        # Step 1: a read tool -> policy will allow it outright.
        {"tool": "lookup_notes", "arguments": {"query": "{prompt}"}},
        # Step 2: an external tool -> policy will pause for approval.
        {"tool": "send_email", "arguments": {"to": "mentor@example.test",
                                             "subject": "Course note",
                                             "body": "{prompt}"}},
    ],
}, indent=2)
print(notes_agent_json)

In [ ]:
# --- Worked solution, part 2: run it ---
# Save it beside the other configurations so load_config() can find it, then run it
# through the SAME runtime object used in Step 1.
from mini_harness import AgentConfig, ModelConfig

raw = json.loads(notes_agent_json)
raw["model"] = ModelConfig(**raw["model"])
notes_agent = AgentConfig(**raw)

result = runtime.run(notes_agent, "harness")

print("Status:", result.status)
print()
print("Policy decisions this new agent produced:")
for event in result.events:
    if event["event"] == "policy_decision":
        d = event["details"]
        print(f"  {d['tool']:<16} risk={d['risk']:<10} -> {d['decision']}")
print()
print("Paused on:", result.pending_action["tool"])
print("Full trace:", [e["event"] for e in result.events])
print()
print("Lines of harness code changed to support a brand-new agent: 0")
print("(To keep it, write notes_agent_json to configs/notes_agent.json and use")
print(" load_config('notes_agent') from then on.)")

### Checkpoint

**1. A colleague says "just add the new agent logic to runtime.py". What is wrong with that?**

<details><summary>Show answer</summary>

The runtime would start containing application-specific behaviour, so every new agent would mean editing shared, safety-critical code. The third agent above needed a JSON document and nothing else - which is exactly why the runtime stayed trustworthy.

</details>

**2. Which responsibility deliberately stays application-specific?**

<details><summary>Show answer</summary>

Which tools an agent may use, and what its instructions are. The harness owns the *mechanism* - discovery, validation, the risk-to-decision table, limits, events. It never decides that this particular agent should be allowed to send email.

</details>

### Recap

- Limitation: reusable infrastructure is only reusable if adding an application needs no change to the infrastructure.
- Layer added: the complete harness - config, provider, registry, policy, runtime, events, checkpoints, memory and an MCP boundary.
- Evidence: three different agents ran through one runtime object, one of them written during the lesson, and the approval pause behaved identically for all of them.
